# Pre-class work: gossip replication, and what replication buys you

## 1. NodeTotalOrderGossip

I subclassed `NodeTotalOrderEagerBroadcast` and changed only how a write spreads.
The ordering half is untouched: Lamport timestamp on every write, `(ts, origin)`
as the order key, a buffer, and a write is delivered once every node's clock has
passed its timestamp.

What changed:

* On first seeing a write, a node sends it to `num_gossip_peers` peers picked at
  random instead of to all of them.
* It repeats that `num_gossip_rounds` times, waiting 100 ms between rounds.
* Each node has its own `random.Random` for picking peers.

The 100 ms gap is the part that matters. A later round is a second attempt at a
write the network may have eaten, and it carries a higher `sender_clock`, which
is what pushes other nodes' `last_seen` forward. If all the rounds fired at once
they would just be extra fanout and would recover nothing.

This class only holds the gossip half, so it needs `NodeTotalOrderEagerBroadcast`
next to it for the Lamport clock, the buffer and the delivery rule.

In [ ]:
import random
from dataclasses import dataclass, field
from typing import Any

from dslabs.nodes.node_total_order_eager_broadcast import NodeTotalOrderEagerBroadcast


@dataclass
class NodeTotalOrderGossip(NodeTotalOrderEagerBroadcast): # I inherited from NodeTotalOrderEagerBroadcast to save time
    """Total order broadcast where replication is gossiped"""

    num_gossip_peers: int = 3
    num_gossip_rounds: int = 3
    gossip_interval_ms: int = 100
    rng: random.Random = field(init=False, repr=False, default=None)

    def __post_init__(self) -> None:
        super().__post_init__()
        self.rng = random.Random()

    def receive_and_replicate(self, key: str, value: Any, ts: int, origin: str) -> None:
        """Buffer a write the first time we see it, then start gossiping it."""
        if (ts, origin) in self.seen:
            return
        self.seen.add((ts, origin))
        self.buffer[(ts, origin)] = (key, value)
        self.gossip(key, value, ts, origin, 0)

    def gossip(self, key: str, value: Any, ts: int, origin: str, round_no: int) -> None:
        others = [p for p in self.peers if p != self.node_id]
        for peer in self.rng.sample(others, min(self.num_gossip_peers, len(others))):
            self.transport.send(peer, {"type": "replicate", "key": key, "value": value,
                                       "ts": ts, "origin": origin, "sender_clock": self.clock})
        if round_no + 1 < self.num_gossip_rounds:
            # Rounds are spread over time on purpose: a later round resends what
            # the network may have dropped, and carries a higher clock with it.
            def next_round() -> None:
                self.gossip(key, value, ts, origin, round_no + 1)

            next_round.description = f"gossip round {round_no + 1} of ({ts}, {origin}) at {self.node_id}"
            self.scheduler.call_later(self.gossip_interval_ms, next_round)

### Setup

`SimSendMany` already gives me the variable network delay (50 to 200 ms) and a
drop probability, so I only vary `drop_prob`, `num_gossip_peers` and
`num_gossip_rounds`.

A run fails if the nodes do not all finish with the same store. That catches both
ways this goes wrong: a write that reached nobody, and a write stuck in a buffer
because some node's clock never moved far enough.

Bandwidth is `SimNetwork.stats["sent"]`, the point to point messages the network
carried.

In [ ]:
from functools import partial
from statistics import mean

from dslabs.nodes import NodeTotalOrderGossip
from dslabs.simulations.sim_send_many import SimSendMany

NUM_NODES = 5
NUM_MESSAGES = 1000
INTERVAL_MS = 100


def run_once(peers, rounds, drop, seed):
    node = partial(NodeTotalOrderGossip, num_gossip_peers=peers, num_gossip_rounds=rounds)
    sim = SimSendMany(node, num_nodes=NUM_NODES, num_messages=NUM_MESSAGES,
                      interval_ms=INTERVAL_MS, drop_prob=drop, seed=seed)
    sim.run(max_ms=2_000_000)
    stores = {tuple(sorted(n.store.items())) for n in sim.nodes.values()}
    stuck = sum(len(n.buffer) for n in sim.nodes.values())
    return len(stores) == 1, sim.network.stats["sent"], stuck

### The experiment

36 settings, 3 runs each, 1000 writes per run. Takes a minute or two. Lower
`REPEATS` or drop a drop rate if that is too slow.

In [ ]:
DROPS = [0.0, 0.2, 0.4, 0.6]
PEERS = [1, 2, 4]
ROUNDS = [1, 2, 3]
REPEATS = 3

results = {}
for drop in DROPS:
    for peers in PEERS:
        for rounds in ROUNDS:
            runs = [run_once(peers, rounds, drop, seed) for seed in range(REPEATS)]
            failures = sum(not ok for ok, _, _ in runs)
            results[(drop, peers, rounds)] = {
                "failure_rate": failures / REPEATS,
                "sent": mean(s for _, s, _ in runs),
                "stuck": mean(k for _, _, k in runs),
            }
    print(f"drop={drop} done")

### Results

In [ ]:
print(f"{'drop':>6}{'peers':>7}{'rounds':>8}{'failure rate':>14}{'messages sent':>15}{'stuck':>8}")
for (drop, peers, rounds), r in results.items():
    print(f"{drop:>6}{peers:>7}{rounds:>8}{r['failure_rate']:>14.2f}"
          f"{r['sent']:>15,.0f}{r['stuck']:>8.1f}")

### Cheapest setting that never failed, per drop rate

This is the answer to which parameters are optimal: no failed run, and the fewest
messages among the settings that managed it.

In [ ]:
print(f"{'drop':>6}{'peers':>7}{'rounds':>8}{'messages sent':>15}")
for drop in DROPS:
    reliable = [(r["sent"], peers, rounds)
                for (d, peers, rounds), r in results.items()
                if d == drop and r["failure_rate"] == 0.0]
    if not reliable:
        print(f"{drop:>6}   nothing in the grid was reliable at this drop rate")
        continue
    sent, peers, rounds = min(reliable)
    print(f"{drop:>6}{peers:>7}{rounds:>8}{sent:>15,.0f}")

### What the numbers say

The two parameters do not do the same job, even though they cost the same.
Bandwidth is close to `NUM_MESSAGES * NUM_NODES * peers * rounds`, so peers 2
rounds 3 and peers 3 rounds 2 send the same number of messages, but they do not
survive the same drop rates.

`num_gossip_peers` is fanout. It decides how far a write gets in one instant and
how many independent paths it takes. It helps when the failure is a write that
never reached some node at all.

`num_gossip_rounds` is time. Each extra round is another attempt after the
network has had its chance to drop the earlier copies, so it is the only one of
the two that recovers a loss instead of trying to outrun it. It is also what
keeps `last_seen` moving, because a node only learns another node's clock by
hearing from that node.

That second job is why rounds matter more than they look. A node cannot deliver
anything until every other node's clock has passed the timestamp, so a setting
that spreads the data fine can still fail, with writes sitting in buffers rather
than lost. The `stuck` column separates the two: writes stuck with nothing lost
is the watermark, not the network.

At drop 0 the extra rounds buy nothing and the cheapest reliable setting wins. As
the drop rate climbs, settings with `rounds=1` start failing whatever the fanout,
because a single instant of sending has no second chance.

This is also the limit of the design. The Lamport watermark needs every node to
be heard from, so gossip here still has to reach everybody eventually, and it
never gets as cheap as gossip usually does.

## 2. What state machine replication buys you

### The application

An online poker room running on several servers. Every server keeps the whole
game state: chip counts, the pot, whose turn it is, and the deck. Players connect
to whichever server is closest. Every action, bet, call, raise or fold, is
broadcast to all the servers, and each one applies the actions in the same order
to the same starting state. Because the state machine is deterministic they all
reach the same result without any of them being in charge. The shuffle comes from
a seed derived from the agreed action log, so no single server has to be trusted
to deal.

### What breaks without it

Without agreement on the order, two servers can disagree about who called first,
so one pays the pot to A and the other to B. Chips are created or destroyed, and
a player who reconnects to a different server sees different money.

Without reliable delivery, a server can miss a fold and sit waiting on a player
who already left, so the table hangs for everyone connected to it.

Without both, the deck diverges. If the shuffle is derived from the log and the
logs differ, servers deal different cards and two players are told they hold the
same ace.

Ordering bugs cost money and reliability bugs cost availability. Poker makes the
first obvious, but any app whose result depends on who acted first has it.

### What broadcast it needs

Using the hierarchy from 5.3, the parts need different strengths.

Player actions need **FIFO total order broadcast**. Total order because the
outcome depends on who acted first and every server has to agree on it. FIFO on
top because one player can send two actions in quick succession and those have to
stay in the order that player sent them, which total order on its own does not
promise.

Table chat needs **causal broadcast**. A reply must not appear before the message
it answers, but two players talking about different things can land in either
order. Total order would work and would be slower for no gain, so causal is the
weakest thing that still reads correctly.

Presence, spectator counts and typing indicators need only **best effort
broadcast**. Losing one does not matter because the next update corrects it.

So the weakest assumption the system can run on is FIFO total order for the
action stream alone. Everything else can be weaker, and putting everything
through total order would add latency to the parts that do not need it.